In [1]:
import pandas as pd
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

import celltypist
import anndata as ad

import seaborn as sns
import matplotlib.pyplot as plt

C:\Users\gupta\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\celltypist\classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np

# Load harmony metadata
meta = pd.read_csv("D:/scRNA/output/harmony_metadata.csv", index_col=0)

# Load UMAP coordinates  
umap = pd.read_csv("D:/scRNA/output/GSE279086_umap_coordinates.csv", index_col=0)

# Create a basic AnnData object from metadata
GSE279086_harmony_adata = ad.AnnData(obs=meta)
GSE279086_harmony_adata.obsm['X_umap'] = umap.loc[GSE279086_harmony_adata.obs_names].values

print(GSE279086_harmony_adata)
print(GSE279086_harmony_adata.obs.columns.tolist())

AnnData object with n_obs × n_vars = 25220 × 0
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'percent.rb', 'Condition', 'mahal_dist_nCount', 'RNA_snn_res.0.8', 'seurat_clusters'
    obsm: 'X_umap'
['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'percent.rb', 'Condition', 'mahal_dist_nCount', 'RNA_snn_res.0.8', 'seurat_clusters']


In [3]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
import pandas as pd
import anndata as ad
import numpy as np

# Load count matrix
counts = mmread("D:/scRNA/output/counts_matrix.mtx").T
counts = csr_matrix(counts)

# Load gene and cell names
genes = pd.read_csv("D:/scRNA/output/gene_names.csv")
cells = pd.read_csv("D:/scRNA/output/cell_names.csv")

# Load metadata
meta = pd.read_csv("D:/scRNA/output/harmony_metadata.csv", index_col=0)

# Load UMAP
umap = pd.read_csv("D:/scRNA/output/GSE279086_umap_coordinates.csv", index_col=0)

# Create AnnData
GSE279086_harmony_adata = ad.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes['gene'].values))
GSE279086_harmony_adata.obsm['X_umap'] = umap.loc[GSE279086_harmony_adata.obs_names].values

print(GSE279086_harmony_adata)

AnnData object with n_obs × n_vars = 25220 × 28317
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'percent.rb', 'Condition', 'mahal_dist_nCount', 'RNA_snn_res.0.8', 'seurat_clusters'
    obsm: 'X_umap'


In [5]:
from celltypist import models
models.download_models(force_update=False)
models.models_path

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 61
📂 Storing models in C:\Users\gupta\.celltypist\data\models
💾 Downloading model [1/61]: Immune_All_Low.pkl
💾 Downloading model [2/61]: Immune_All_High.pkl
💾 Downloading model [3/61]: Adult_COVID19_PBMC.pkl
💾 Downloading model [4/61]: Adult_CynomolgusMacaque_Hippocampus.pkl
💾 Downloading model [5/61]: Adult_Human_MTG.pkl
💾 Downloading model [6/61]: Adult_Human_PancreaticIslet.pkl
💾 Downloading model [7/61]: Adult_Human_PrefrontalCortex.pkl
💾 Downloading model [8/61]: Adult_Human_Skin.pkl
💾 Downloading model [9/61]: Adult_Human_Vascular.pkl
💾 Downloading model [10/61]: Adult_Mouse_Gut.pkl
💾 Downloading model [11/61]: Adult_Mouse_OlfactoryBulb.pkl
💾 Downloading model [12/61]: Adult_Pig_Hippocampus.pkl
💾 Downloading model [13/61]: Adult_RhesusMacaque_Hippocampus.pkl
💾 Downloading model [14/61]: Adult_cHSPCs_Illumina.pkl
💾 Downloading model [15/61]: Adult_cHSPCs_Ultima.pkl
💾 

'C:\\Users\\gupta\\.celltypist\\data\\models'

In [6]:
import celltypist
import scanpy as sc

sc.pp.normalize_total(GSE279086_harmony_adata, target_sum=1e4)
sc.pp.log1p(GSE279086_harmony_adata)

ct_pred = celltypist.annotate(GSE279086_harmony_adata, 
                               model="Immune_All_Low.pkl", 
                               majority_voting=True)

GSE279086_harmony_adata.obs['majority_voting'] = ct_pred.predicted_labels['majority_voting'].astype(str)

print(GSE279086_harmony_adata.obs['majority_voting'].value_counts().head(10))

🔬 Input data has 25220 cells and 28317 genes
🔗 Matching reference genes in the model
🧬 5759 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 15
🗳️ Majority voting the predictions
✅ Majority voting done!


majority_voting
Epithelial cells                 15053
Endothelial cells                 5703
Erythrophagocytic macrophages     2150
Tcm/Naive helper T cells           800
Memory B cells                     455
Non-classical monocytes            321
CD16+ NK cells                     227
DC2                                130
Fibroblasts                        111
Tem/Temra cytotoxic T cells        109
Name: count, dtype: int64


In [7]:
GSE279086_harmony_adata.obs.to_csv("D:/scRNA/output/celltypist_annotations.csv")
print("Annotations saved!")

Annotations saved!


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc

# Load Seurat UMAP coordinates
umap_df = pd.read_csv("c:/Users/smriti/Desktop/demo/WGSE279086/W2/GSE279086_umap_coordinates.csv", index_col=0)

# Sanity check
print(umap_df.shape) 

# Inject into AnnData
GSE279086_harmony_adata.obsm['X_umap'] = umap_df.loc[GSE279086_harmony_adata.obs_names].values
sc.pl.umap(GSE279086_harmony_adata, color='seurat_clusters', show=True)

In [ ]:
# Put log-normalized counts into X (only for CellTypist)
GSE279086_harmony_adata.X = GSE279086_harmony_adata.layers['logcounts'].copy()

In [ ]:
import celltypist
from celltypist import models

ct_pred = celltypist.annotate(GSE279086_harmony_adata, model="c:/Users/smriti/Desktop/demo/WGSE279086/Adult_Human_Kidney.pkl", majority_voting=True)

In [ ]:
GSE279086_harmony_adata.obs['majority_voting'] = (ct_pred.predicted_labels['majority_voting'].astype(str))

In [ ]:
print(GSE279086_harmony_adata.obs['majority_voting'].value_counts().head())

In [ ]:
import matplotlib.pyplot as plt

for cond in GSE279086_harmony_adata.obs['Condition'].unique():
    fig = sc.pl.umap(GSE279086_harmony_adata[GSE279086_harmony_adata.obs['Condition'] == cond],
        color='majority_voting',title=f'Condition: {cond}',legend_loc='right margin',show=False,return_fig=True)
    plt.savefig(f"C:/users/smriti/Desktop/demo/WGSE279086/W2/plots/ct_{cond}.png",dpi=300,bbox_inches='tight')
    plt.show()

In [ ]:
import celltypist
from celltypist import models

# Put log-normalized counts into X (only for CellTypist)
GSE279086_raw_adata.X = GSE279086_raw_adata.layers['logcounts'].copy()
ct_pred = celltypist.annotate(GSE279086_raw_adata, model="c:/Users/smriti/Desktop/demo/WGSE279086/Adult_Human_Kidney.pkl", majority_voting=True)

In [ ]:
GSE279086_raw_adata.obs['majority_voting'] = (ct_pred.predicted_labels['majority_voting'].astype(str))

In [ ]:
print(GSE279086_raw_adata.obs.columns.tolist())

In [ ]:
# Step 1: Count cells per cell type per condition
n_cells_condition = (GSE279086_raw_adata.obs.groupby(["Condition", "majority_voting"]).size().reset_index(name="count"))

# Step 2: Calculate proportions per condition
n_cells_condition["total"] = n_cells_condition.groupby("Condition")["count"].transform("sum")
n_cells_condition["proportion"] = (n_cells_condition["count"] / n_cells_condition["total"]) * 100

# Step 3: Compute average proportion per cell type for ordering
avg_proportions = (n_cells_condition.groupby("majority_voting")["proportion"].mean().sort_values(ascending=False))
ordered_celltypes = avg_proportions.index.tolist()

# Step 4: Plot
plt.figure(figsize=(40, 10))
ax = sns.barplot(
    data=n_cells_condition,
    x="majority_voting",
    y="proportion",
    hue="Condition",
    order=ordered_celltypes,
    dodge=True,
    width=0.9
)

# Step 5: Add proportion labels
for container in ax.containers:ax.bar_label(container, fmt='%.1f%%', label_type='edge', padding=2, fontsize=10, rotation = 90)

# Customize axes
plt.xticks(rotation=45, ha="right", fontsize=12, fontweight="bold")
plt.ylabel("Proportion (%)", fontsize=12, fontweight="bold")
plt.xlabel("Cell Type", fontsize=12, fontweight="bold")
plt.title("Cell Type Proportions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("c:/Users/smriti/Desktop/demo/WGSE279086/W2/plots/cellproportions_barplot.png", dpi=300, bbox_inches='tight')
plt.show()

n_cells_condition.to_csv("c:/users/smriti/Desktop/demo/WGSE279086/W2/output/Cellproportions.csv", index=False)

In [ ]:
GSE279086_raw_adata.write("C:/users/smriti/Desktop/demo/WGSE279086/W2/output/GSE279086_celltypist.h5ad")